# Synthetic datasets

# Exercise 2 (EDA)

## Part 1 (structuring)

In [9]:
import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta

# Reproducibility
np.random.seed(42)
random.seed(42)

# Expanded company list
companies = [
    ("SAP", "DE", "Tech"), ("Siemens", "DE", "Industrial"),
    ("Allianz", "DE", "Finance"), ("BMW", "DE", "Automotive"),
    ("Apple", "US", "Tech"), ("Microsoft", "US", "Tech"),
    ("Amazon", "US", "Tech"), ("Tesla", "US", "Automotive"),
    ("GoldmanSachs", "US", "Finance"), ("JPMorgan", "US", "Finance"),
    ("Sony", "JP", "Tech"), ("Toyota", "JP", "Automotive"),
    ("SoftBank", "JP", "Finance"),
    ("Samsung", "KR", "Tech"), ("Hyundai", "KR", "Automotive"),
    ("Tencent", "CN", "Tech"), ("Alibaba", "CN", "Tech"),
    ("ICBC", "CN", "Finance"),
    ("Shell", "UK", "Energy"), ("BP", "UK", "Energy"),
    ("HSBC", "UK", "Finance"),
    ("TotalEnergies", "FR", "Energy"), ("LVMH", "FR", "Luxury"),
    ("Airbus", "FR", "Industrial"),
    ("Nestle", "CH", "Consumer"), ("Novartis", "CH", "Healthcare"),
    ("Roche", "CH", "Healthcare"),
]

aggregates = [
    ("Total", "US", "Market"),
    ("Total", "EU", "Market"),
    ("Total", "APAC", "Market"),
    ("Global", "ALL", "Market")
]

entities = companies + aggregates

# Periods
periods = [
    "2023Q1", "2023Q2", "2023Q3", "2023Q4",
    "2024Q1", "2024Q2", "2024Q3", "2024Q4"
]

metrics = ["MarketCap", "Revenue", "EBITDA"]

records = []

# Timestamp range
start_date = datetime(2023, 1, 1)
end_date = datetime(2024, 12, 31)
date_range_days = (end_date - start_date).days

for entity, country, sector in entities:
    for period in periods:

        # ✅ ONE timestamp per firm-period
        random_days = random.randint(0, date_range_days)
        timestamp = start_date + timedelta(days=random_days)
        timestamp_str = timestamp.strftime("%Y-%m-%d %H:%M:%S")

        # Generate all metric values first
        values = {}
        for metric in metrics:
            if metric == "MarketCap":
                val = np.random.uniform(50, 3000)
            elif metric == "Revenue":
                val = np.random.uniform(5, 500)
            else:
                val = np.random.uniform(1, 200)
            values[metric] = f"{round(val, 1)}B"

        # Create one row per metric (still messy!)
        for metric in metrics:
            record = f"{entity}-{country}-{sector}-{period}-{metric}-{values[metric]}"

            records.append({
                "timestamp": timestamp_str,  # ✅ shared timestamp
                "logger_sys": "FIN-PIPE-V1",
                "Record": record
            })

# Add extra samples to reach ~1000
extra_samples = random.choices(records, k=250)
records.extend(extra_samples)

# Create DataFrame
df = pd.DataFrame(records)

# Save
df.to_csv("market_data_log.csv", index=False)

# Preview
df.head()

,timestamp,logger_sys,Record
0,2024-10-16 00:00:00,FIN-PIPE-V1,SAP-DE-Tech-2023Q1-MarketCap-1154.9B
1,2024-10-16 00:00:00,FIN-PIPE-V1,SAP-DE-Tech-2023Q1-Revenue-475.6B
2,2024-10-16 00:00:00,FIN-PIPE-V1,SAP-DE-Tech-2023Q1-EBITDA-146.7B
3,2023-04-25 00:00:00,FIN-PIPE-V1,SAP-DE-Tech-2023Q2-MarketCap-1816.0B
4,2023-04-25 00:00:00,FIN-PIPE-V1,SAP-DE-Tech-2023Q2-Revenue-82.2B


## Part 2 (structuring)

In [2]:
import pandas as pd
import numpy as np
import random
import string

# Set seed for reproducibility
np.random.seed(42)
random.seed(42)

def generate_customer_id():
    prefix = random.choice(["CUST", "CL", "ACC"])
    suffix = ''.join(random.choices(string.ascii_uppercase + string.digits, k=6))
    return f"{prefix}-{suffix}"

# Parameters
n_customers = 100
customer_ids = [generate_customer_id() for _ in range(n_customers)]
regions = np.random.choice(["EU", "US", "APAC"], size=n_customers)

months = ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
          "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
channels = ["Online", "Store"]

# Base DataFrame
df = pd.DataFrame({
    "Customer_ID": customer_ids,
    "Region": regions
})

# Generate sales data dynamically
for month in months:
    for channel in channels:
        base_low = 80 if channel == "Online" else 50
        base_high = 300 if channel == "Online" else 250
        
        df[f"{month}_{channel}"] = np.random.randint(
            base_low, base_high, size=n_customers
        )

# Optional: introduce slight regional patterns
online_cols = [col for col in df.columns if "Online" in col]
store_cols = [col for col in df.columns if "Store" in col]

df.loc[df["Region"] == "EU", online_cols] += 20
df.loc[df["Region"] == "US", store_cols] += 15

# Save to CSV
df.to_csv("sales_data_channels_full_year.csv", index=False)

# Preview
df.head()

# possible extensions
# missing values
# df.loc[np.random.choice(df.index, 5), random.choice(df.columns[2:])] = np.nan

,Customer_ID,Region,Jan_Online,Jan_Store,Feb_Online,Feb_Store,Mar_Online,Mar_Store,Apr_Online,Apr_Store,...,Aug_Online,Aug_Store,Sep_Online,Sep_Store,Oct_Online,Oct_Store,Nov_Online,Nov_Store,Dec_Online,Dec_Store
0,ACC-E0IFD0,APAC,215,201,240,73,80,184,224,240,...,182,180,108,154,278,76,85,211,148,132
1,ACC-DPBHSA,EU,315,108,251,163,133,245,170,194,...,300,157,289,233,173,121,172,213,260,158
2,CUST-ZZPQK5,APAC,142,167,195,81,175,107,296,210,...,101,221,141,116,142,109,182,55,258,124
3,CUST-ZMF8MD,APAC,218,209,154,224,205,163,124,206,...,83,228,264,172,235,223,239,88,285,178
4,CUST-MMJBQE,EU,180,145,212,135,217,124,231,190,...,253,151,115,78,159,194,140,219,111,137


## Part 3 (cleansing)

In [3]:
import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta

# Reproducibility
np.random.seed(42)
random.seed(42)

n = 2000

# ---------------------------
# Helper functions
# ---------------------------

def generate_customer_id(i):
    return f"CUST_{str(i).zfill(11)}"

def random_dob():
    start = datetime(1940, 1, 1)
    end = datetime(2005, 12, 31)
    delta = (end - start).days
    return start + timedelta(days=random.randint(0, delta))

# ---------------------------
# Base clean data
# ---------------------------

customer_ids = [generate_customer_id(i) for i in range(1, n+1)]

countries = [
    "DE", "Germany", "US", "USA", "UK", "France", "FR",
    "Spain", "ES", "Italy", "IT", "Netherlands", "NL",
    "Switzerland", "CH", "Austria", "AT",
    "China", "CN", "Japan", "JP", "India", "IN",
    "Brazil", "BR", "Canada", "CA"
]

genders = ["M", "F"]

df = pd.DataFrame({
    "customer_id": customer_ids,
    "date_of_birth": [random_dob().strftime("%Y-%m-%d") for _ in range(n)],
    "income": np.random.randint(2000, 8000, size=n).astype(str),
    "gender": np.random.choice(genders, size=n),
    "signup_date": pd.date_range(start="2022-01-01", periods=n).astype(str),
    "country": np.random.choice(countries, size=n)
})

# ---------------------------
# Introduce data quality issues
# ---------------------------

# 1. Missing values
df.loc[np.random.choice(df.index, 100), "date_of_birth"] = None
df.loc[np.random.choice(df.index, 100), "income"] = None
df.loc[np.random.choice(df.index, 50), "signup_date"] = None

# 2. Implausible values (DOB in future / too old)
df.loc[np.random.choice(df.index, 10), "date_of_birth"] = "1890-01-01"
df.loc[np.random.choice(df.index, 10), "date_of_birth"] = "2030-01-01"

# 3. Inconsistent income formats
idx = np.random.choice(df.index, 200, replace=False)
df.loc[idx, "income"] = df.loc[idx, "income"] + " EUR"

idx = np.random.choice(df.index, 50, replace=False)
df.loc[idx, "income"] = df.loc[idx, "income"] + " USD"

# 4. Outliers in income
df.loc[np.random.choice(df.index, 10), "income"] = ["-5000", "200000"] * 5

# 5. Inconsistent gender formats
df.loc[np.random.choice(df.index, 100), "gender"] = np.random.choice(
    ["Male", "Female", "m", "f"], size=100
)

# 6. Inconsistent date formats
idx = np.random.choice(df.index, 100, replace=False)
df.loc[idx, "signup_date"] = pd.to_datetime(
    df.loc[idx, "signup_date"], errors="coerce"
).dt.strftime("%d/%m/%y")

idx = np.random.choice(df.index, 50, replace=False)
df.loc[idx, "signup_date"] = pd.to_datetime(
    df.loc[idx, "signup_date"], errors="coerce"
).dt.strftime("%Y/%m/%d")

# 7. Invalid dates
df.loc[np.random.choice(df.index, 5), "signup_date"] = "2023-13-01"

# 8. Country inconsistencies / typos
df.loc[np.random.choice(df.index, 50), "country"] = np.random.choice(
    ["Ger", "DEU", "U.S.", "UK ", "Munch", "Brasil"], size=50
)

# 9. Duplicate records
duplicates = df.sample(50, random_state=42)
df = pd.concat([df, duplicates], ignore_index=True)

# Shuffle dataset
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

# Save
df.to_csv("messy_customer_data.csv", index=False)

# Preview
df.head()

,customer_id,date_of_birth,income,gender,signup_date,country
0,CUST_00000001809,1980-04-09,4896 USD,M,2026-12-14,USA
1,CUST_00000000695,1945-08-02,7029,F,2023-11-26,BR
2,CUST_00000000907,1971-07-25,6432,F,2024-06-25,Austria
3,CUST_00000000545,2000-03-01,6107,M,2023-06-29,JP
4,CUST_00000001848,1943-09-06,7655,F,2027-01-22,Italy


## Part 4 (clustering)

In [4]:
import pandas as pd
import numpy as np

np.random.seed(42)

n_samples = 200

# Cluster 1: High-growth tech firms
cluster_1 = pd.DataFrame({
    "revenue": np.random.normal(500, 100, n_samples//4),
    "profit_margin": np.random.normal(5, 3, n_samples//4),
    "growth_rate": np.random.normal(20, 5, n_samples//4),
    "debt_ratio": np.random.normal(0.3, 0.1, n_samples//4),
    "market_cap": np.random.normal(5000, 1000, n_samples//4),
    "rd_intensity": np.random.normal(15, 5, n_samples//4),
})

# Cluster 2: Mature stable firms
cluster_2 = pd.DataFrame({
    "revenue": np.random.normal(2000, 500, n_samples//4),
    "profit_margin": np.random.normal(15, 5, n_samples//4),
    "growth_rate": np.random.normal(5, 2, n_samples//4),
    "debt_ratio": np.random.normal(0.5, 0.1, n_samples//4),
    "market_cap": np.random.normal(15000, 3000, n_samples//4),
    "rd_intensity": np.random.normal(5, 2, n_samples//4),
})

# Cluster 3: Distressed firms
cluster_3 = pd.DataFrame({
    "revenue": np.random.normal(800, 200, n_samples//4),
    "profit_margin": np.random.normal(-5, 5, n_samples//4),
    "growth_rate": np.random.normal(-2, 3, n_samples//4),
    "debt_ratio": np.random.normal(0.8, 0.1, n_samples//4),
    "market_cap": np.random.normal(2000, 800, n_samples//4),
    "rd_intensity": np.random.normal(2, 1, n_samples//4),
})

# Cluster 4: Innovation-heavy niche firms
cluster_4 = pd.DataFrame({
    "revenue": np.random.normal(300, 80, n_samples//4),
    "profit_margin": np.random.normal(8, 4, n_samples//4),
    "growth_rate": np.random.normal(12, 4, n_samples//4),
    "debt_ratio": np.random.normal(0.4, 0.1, n_samples//4),
    "market_cap": np.random.normal(4000, 1200, n_samples//4),
    "rd_intensity": np.random.normal(25, 7, n_samples//4),
})

# Combine dataset
df_finance = pd.concat([cluster_1, cluster_2, cluster_3, cluster_4], ignore_index=True)

# Optional: add noise
df_finance += np.random.normal(0, 0.05 * df_finance.std(), df_finance.shape)

df_finance["sector"] = np.random.choice(
    ["Tech", "Finance", "Healthcare", "Industry"],
    size=len(df_finance)
)
df_finance["company_id"] = ["FIRM-" + str(i).zfill(4) for i in range(len(df_finance))]
df_finance = df_finance.sample(frac=1, random_state=42).reset_index(drop=True)

cols = [
    "company_id",
    "sector",
    "revenue",
    "profit_margin",
    "growth_rate",
    "debt_ratio",
    "market_cap",
    "rd_intensity"
]

df_finance = df_finance[cols]

# Save
df_finance.to_csv("firm_data.csv", index=False)

# Preview
df_finance.head()

,company_id,sector,revenue,profit_margin,growth_rate,debt_ratio,market_cap,rd_intensity
0,FIRM-0095,Healthcare,2117.569743,13.382162,6.053507,0.555974,12426.889897,3.308879
1,FIRM-0015,Healthcare,457.538045,10.139487,21.504950,0.350252,5779.074618,8.710809
2,FIRM-0030,Industry,463.847307,4.131422,13.237034,0.356084,4451.965515,15.719744
3,FIRM-0158,Finance,347.980562,9.573304,16.117482,0.472141,3256.609038,31.989645
4,FIRM-0128,Tech,804.239654,0.165357,-3.346796,0.761088,1825.004761,2.082579
